# Fine-tuning de LLMs con LoRA (PEFT)
Modelo: `bigscience/bloomz-560m` · Dataset: `Abirate/english_quotes` (10% del split de train)

⚠️ Usa GPU: *Entorno de ejecución → Cambiar tipo de entorno → T4 GPU*.

## 1 Install necessary libraries (PEFT, datasets).

In [1]:
!pip install -q peft datasets accelerate
!mkdir -p cache


[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2 Load a pre-trained language model (bigscience/bloomz-560m) and its tokenizer.

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "bigscience/bloomz-560m"
tokenizer = AutoTokenizer.from_pretrained(model_name)
foundation_model = AutoModelForCausalLM.from_pretrained(model_name)

# 10% trai...
data = load_dataset("Abirate/english_quotes", split="train[:10%]")
data = data.map(lambda samples: tokenizer(samples["quote"]), batched=True)

train_sample = data.select(range(5))
display(train_sample)

c:\Users\Juan Serebrinsky\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3 Configure LoRA using LoraConfig.

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=1,                                   # range
    lora_alpha=1,                          # factor
    target_modules=["query_key_value"],    # atention bloom
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

peft_model = get_peft_model(foundation_model, lora_config)
peft_model.print_trainable_parameters()

## 4train

In [ ]:
import os
import transformers
from transformers import TrainingArguments, Trainer

output_directory = os.path.join("cache", "working", "peft_lab_outputs")

training_args = TrainingArguments(
    report_to="none",
    output_dir=output_directory,
    auto_find_batch_size=True,
    learning_rate=3e-2,        
    num_train_epochs=3,
)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=data,
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False),
)
trainer.train()

## 5. Guardar el adaptador LoRA

In [ ]:
import time

time_now = time.strftime("%Y%m%d-%H%M%S")
peft_model_path = os.path.join(output_directory, f"peft_model_{time_now}")
trainer.model.save_pretrained(peft_model_path)
print("Guardado en:", peft_model_path)

## 6 Load the saved LoRA model for inference using PeftModel.from_pretrained.

In [ ]:
from peft import PeftModel

# load clean and saved
base_model = AutoModelForCausalLM.from_pretrained(model_name)
loaded_model = PeftModel.from_pretrained(base_model, peft_model_path, is_trainable=False)
loaded_model.to("cuda" if torch.cuda.is_available() else "cpu")

## 7 Generate text using the fine-tuned model and the tokenizer.

In [ ]:
inputs = tokenizer("Two things are infinite: ", return_tensors="pt").to(loaded_model.device)

outputs = loaded_model.generate(
    input_ids=inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    max_new_tokens=50,
    do_sample=True,
    top_p=0.9,
    temperature=0.8,
    repetition_penalty=1.2,
    pad_token_id=tokenizer.eos_token_id,
)

print(tokenizer.batch_decode(outputs, skip_special_tokens=True))